In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans

# Visual style for your report graphs
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Function to chop DNA into 4-letter motifs (k-mers)
def get_kmers(sequence, k=4):
    return [sequence[i:i+k].upper() for i in range(len(sequence) - k + 1)]

# Dummy analyzer to save memory
def dummy_analyzer(doc):
    return doc

# NOTE: Replace this with your actual CSV file path when you have it!
# df = pd.read_csv('ceroni_120k_plasmids.csv')

# --- Dummy Data for Testing (Deletes this block later) ---
np.random.seed(42)
sequences = [''.join(np.random.choice(['A', 'T', 'G', 'C'], size=200)) for _ in range(1000)]
fitness_scores = np.random.normal(loc=0.5, scale=0.2, size=1000)
df = pd.DataFrame({'sequence': sequences, 'fitness': fitness_scores})
# ---------------------------------------------------------

print("Extracting motifs from DNA...")
df['kmers'] = df['sequence'].apply(lambda x: get_kmers(x, k=4))

In [ ]:
print("Converting DNA to math (Vectorizing)...")
vectorizer = CountVectorizer(analyzer=dummy_analyzer)
X_sparse = vectorizer.fit_transform(df['kmers'])
feature_names = vectorizer.get_feature_names_out()

print("Squashing dimensions (TruncatedSVD)...")
svd = TruncatedSVD(n_components=2, random_state=42)
X_reduced = svd.fit_transform(X_sparse)

df['PC1'] = X_reduced[:, 0]
df['PC2'] = X_reduced[:, 1]

print("Finding patterns (Clustering)...")
kmeans = MiniBatchKMeans(n_clusters=3, batch_size=1024, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_sparse)

In [ ]:
# Plotting the results to show your supervisors
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Graph 1: Actual biological fitness
scatter1 = ax1.scatter(df['PC1'], df['PC2'], c=df['fitness'], cmap='coolwarm', alpha=0.6, s=10)
fig.colorbar(scatter1, ax=ax1, label='Metabolic Burden (Fitness)')
ax1.set_title('Plasmid Sequences by Actual Fitness')

# Graph 2: The computer's unsupervised clusters
scatter2 = ax2.scatter(df['PC1'], df['PC2'], c=df['Cluster'], cmap='viridis', alpha=0.6, s=10)
fig.colorbar(scatter2, ax=ax2, label='Computer Generated Cluster')
ax2.set_title('Plasmid Sequences by Mathematical Cluster')

plt.show()

In [ ]:
# Extracting the actual bad sequences (Mechanistic Interpretation)
loadings_pc1 = svd.components_[0]
motifs_df = pd.DataFrame({'Motif': feature_names, 'Importance': loadings_pc1})

print("Top 10 DNA Motifs driving the variance:")
print(motifs_df.sort_values(by='Importance', ascending=False).head(10))